In [1]:

from pathlib import Path
import re, json, shutil, subprocess
import os

In [34]:

def _natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def renumber_files_macname(
    folder_path,
    dry_run=True,
    digits=2,
    skip_hidden=True,
):
    folder = Path(folder_path)

    files = []
    for f in folder.iterdir():
        if not f.is_file():
            continue
        if skip_hidden and f.name.startswith("."):
            continue
        files.append(f)

    def cleaned_for_sort(name: str) -> str:
        # 排序时忽略已有编号前缀
        return re.sub(r"^\d+_", "", name)

    files.sort(key=lambda p: _natural_key(cleaned_for_sort(p.name)))

    print("\n=== ORDER (filtered, ignore old prefix) ===")
    for f in files:
        print(cleaned_for_sort(f.name))

    plan = []
    for idx, f in enumerate(files, start=1):
        cleaned = cleaned_for_sort(f.name)
        new_name = f"{idx:0{digits}d}_{cleaned}"
        plan.append((f, new_name))

    print("\n=== RENAME PLAN ===")
    for f, new_name in plan:
        print(f"{f.name}  →  {new_name}")

    if dry_run:
        print("\n[DRY RUN] No files were renamed.")
        return

    # 两步改名避免撞名
    temp_plan = []
    for i, (f, new_name) in enumerate(plan, start=1):
        tmp_name = f".__tmp__{i:05d}__{f.name}"
        tmp_path = folder / tmp_name
        f.rename(tmp_path)
        temp_plan.append((tmp_path, folder / new_name))

    for tmp_path, final_path in temp_plan:
        tmp_path.rename(final_path)

    print("\nRenaming complete.")

renumber_files_macname("audio", dry_run=True)
# renumber_files_macname("audio", dry_run=False)


=== ORDER (filtered, ignore old prefix) ===
第1季_第1集_满腹猪鸡.m4a
第1季_第2集_惊鸿一瞥.m4a
第1季_第3集_心肝乖乖.m4a
第1季_第4集_林中学射.m4a
第1季_第5集_臣委屈（哭）.m4a
第1季_第6集_门柜床梁.m4a
第1季_第7集_临危受托.m4a
第1季_第8集_梳洗之刑.m4a
第1季_第9集_好戏开场.m4a
第1季_第10集_看上你了.m4a
第1季_第11集_小妾风波.m4a
第1季_第12集_生辰加冠.m4a
第1季_第13集_祸起灵光.m4a
第1季_第14集_水榭纠缠.m4a
第1季_第15集_临别送行.m4a
第1季_第16集_心念紫宸.m4a
第1季_第17集_天远地阔.m4a
第1季_第18集_病入膏肓.m4a
第1季_第19集_愿者上钩.m4a
第1季_第20集（上）_贪心一刻.m4a
第1季_第20集（下）_罡风四起.m4a

=== RENAME PLAN ===
01_第1季_第1集_满腹猪鸡.m4a  →  01_第1季_第1集_满腹猪鸡.m4a
02_第1季_第2集_惊鸿一瞥.m4a  →  02_第1季_第2集_惊鸿一瞥.m4a
03_第1季_第3集_心肝乖乖.m4a  →  03_第1季_第3集_心肝乖乖.m4a
04_第1季_第4集_林中学射.m4a  →  04_第1季_第4集_林中学射.m4a
05_第1季_第5集_臣委屈（哭）.m4a  →  05_第1季_第5集_臣委屈（哭）.m4a
06_第1季_第6集_门柜床梁.m4a  →  06_第1季_第6集_门柜床梁.m4a
07_第1季_第7集_临危受托.m4a  →  07_第1季_第7集_临危受托.m4a
08_第1季_第8集_梳洗之刑.m4a  →  08_第1季_第8集_梳洗之刑.m4a
09_第1季_第9集_好戏开场.m4a  →  09_第1季_第9集_好戏开场.m4a
10_第1季_第10集_看上你了.m4a  →  10_第1季_第10集_看上你了.m4a
11_第1季_第11集_小妾风波.m4a  →  11_第1季_第11集_小妾风波.m4a
12_第1季_第12集_生辰加冠.m4a  →  12_第1季_第12集_生辰加冠.m4a
13_第1季_第13集_祸起灵光.m4

In [36]:

def add_to_prefix_number(folder_path, x, dry_run=True):
    folder = Path(folder_path)

    print("\n=== RENAME PLAN ===")

    for f in folder.iterdir():
        if not f.is_file():
            continue
        
        # 跳过 DS_Store
        if ".DS_Store" in f.name:
            continue

        # 匹配开头数字_
        match = re.match(r'^(\d+)_', f.name)
        if not match:
            continue  # 没有数字前缀就跳过

        old_number = int(match.group(1))
        new_number = old_number + x

        # 保持原位数（比如 01 → 03）
        width = len(match.group(1))
        new_prefix = f"{new_number:0{width}d}_"

        # 替换旧前缀
        new_name = re.sub(r'^\d+_', new_prefix, f.name)

        print(f"{f.name}  →  {new_name}")

        if not dry_run:
            f.rename(folder / new_name)

    if dry_run:
        print("\n[DRY RUN] No files renamed.")
    else:
        print("\nRenaming complete.")

add_to_prefix_number("add", x=3, dry_run=True)
# add_to_prefix_number("add", x=3, dry_run=False)


=== RENAME PLAN ===
30_小剧场_雨后风荷.m4a  →  33_小剧场_雨后风荷.m4a
58_花絮1.m4a  →  61_花絮1.m4a
52_800w福利_与君书_手书.m4a  →  55_800w福利_与君书_手书.m4a
27_小剧场_假神仙.m4a  →  30_小剧场_假神仙.m4a
31_冬至特别语音.m4a  →  34_冬至特别语音.m4a
47_500w特别剧场_罄露_情动.m4a  →  50_500w特别剧场_罄露_情动.m4a
59_花絮2.m4a  →  62_花絮2.m4a
35_元宵特别语音.m4a  →  38_元宵特别语音.m4a
53_800w福利_与君书_批注.m4a  →  56_800w福利_与君书_批注.m4a
45_500w特别剧场_罄露_同榻.m4a  →  48_500w特别剧场_罄露_同榻.m4a
37_祝苏大人生辰康乐.m4a  →  40_祝苏大人生辰康乐.m4a
65_特别花絮.m4a  →  68_特别花絮.m4a
43_300w福利_得闲_断袖.m4a  →  46_300w福利_得闲_断袖.m4a
56_800w福利_与君书_纸条.m4a  →  59_800w福利_与君书_纸条.m4a
49_500w特别剧场_罄露_邀吻.m4a  →  52_500w特别剧场_罄露_邀吻.m4a
28_小剧场_冯党苏党？.m4a  →  31_小剧场_冯党苏党？.m4a
32_元旦特别语音.m4a  →  35_元旦特别语音.m4a
54_800w福利_与君书_日记.m4a  →  57_800w福利_与君书_日记.m4a
26_小剧场_小爷难缠.m4a  →  29_小剧场_小爷难缠.m4a
64_花絮7.m4a  →  67_花絮7.m4a
25_小剧场_鸳鸯腿.m4a  →  28_小剧场_鸳鸯腿.m4a
38_100W福利音_小憩.m4a  →  41_100W福利音_小憩.m4a
60_花絮3.m4a  →  63_花絮3.m4a
33_除夕特别剧场_“朱”你新年快乐.m4a  →  36_除夕特别剧场_“朱”你新年快乐.m4a
61_花絮4.m4a  →  64_花絮4.m4a
50_800w福利_与君书_便笺.m4a  →  53_800w福利_与君书_便笺.m4a
48_

In [49]:


# 再世权臣_天谢_苏晏/倒霉死勒_朱栩竟/顺子_朱槿隚/赵成晨_沈柒/马正阳_阿勒坦/余昊威_荆红追/邓宥希_朱贺霖/余昌宇
ALBUM      = "再世权臣_第2季_天谢" # 专辑/系列名（Option A/B 都用这个）
BOOK_TITLE = ALBUM      # Option B 的大文件 Title
AUTHOR     = "苏晏/倒霉死勒_朱栩竟/顺子_朱槿隚/赵成晨_沈柒/马正阳_阿勒坦/余昊威_荆红追/邓宥希_朱贺霖/余昌宇"          # 作者/播音
COVER      = Path("cover.jpg")  # 封面文件名固定 cover.jpg
INPUT_DIR  = Path("audio")      # 音频文件夹

# =======================

assert INPUT_DIR.exists(), "找不到 audio 文件夹，请确认 notebook 的工作目录"
assert COVER.exists(), "找不到 cover.jpg，请把封面放在 notebook 同级目录"

def which(cmd):
    p = shutil.which(cmd)
    if not p:
        raise RuntimeError(f"未找到 {cmd}，请先安装：brew install ffmpeg")
    return p

FFMPEG = which("ffmpeg")
FFPROBE = which("ffprobe")

def sort_key(p: Path):
    name = p.stem
    
    # 取文件名前面的数字
    m = re.match(r"^\d+", name)
    if m:
        return int(m.group())
    
    # 如果没有数字就放最后
    return float("inf")

files = sorted(INPUT_DIR.glob("*.m4a"), key=sort_key)
assert files, "audio/ 里没找到 .m4a 文件"

print(f"Found {len(files)} files:")
for f in files:
    print(" -", f.name)

Found 41 files:
 - 01_第2季_第1集（上）_非我族类.m4a
 - 02_第2季_第1集（下）_追根究底.m4a
 - 03_第2季_第2集_相思无尽.m4a
 - 04_第2季_第3集_好梦一晌.m4a
 - 05_第2季_第4集_再（不）谐鱼水.m4a
 - 06_第2季_第5集_梅仙衔梅.m4a
 - 07_第2季_第6集_一厢情愿.m4a
 - 08_第2季_第7集_攀枝窃香.m4a
 - 09_第2季_第8集_恕难从命.m4a
 - 10_第2季_第9集_佛闻弃禅.m4a
 - 11_第2季_第10集_牵钩之戏.m4a
 - 12_第2季_第11集_海晏河清.m4a
 - 13_第2季_第12集_白首不离.m4a
 - 14_第2季_第13集_断袖重接（？）.m4a
 - 15_第2季_第14集_出师未捷.m4a
 - 16_第2季_第15集_顺应天意.m4a
 - 17_第2季_第16集_甘为锁链.m4a
 - 18_第2季_第17集_无名战神.m4a
 - 19_第2季_第18集_怒斥群臣.m4a
 - 20_第2季_第19集_道之所在.m4a
 - 21_第2季_第20集（上）_暂时结盟.m4a
 - 22_第2季_第20集（下）_暗流再起.m4a
 - 23_倒计时3天_爬到上面.m4a
 - 24_倒计时2天_似曾相识.m4a
 - 25_倒计时1天_黄色废料.m4a
 - 26_小剧场_好小妾.m4a
 - 27_小剧场_鸿雁传书.m4a
 - 28_小剧场_鳌山灯会.m4a
 - 29_小剧场_后爹在哪里.m4a
 - 30_小剧场_怪哉怪哉.m4a
 - 31_冬至特别语音_饺子也攻斗.m4a
 - 32_元旦特别来电.m4a
 - 33_除夕特别剧场_饺子饺子饺子.m4a
 - 34_除夕特别剧场_大铭有春晚.m4a
 - 35_元宵特别语音.m4a
 - 36_100W福利音_早起.m4a
 - 37_200W福利音_苏相更新了朋友圈.m4a
 - 38_400W福利音_苏相更新了朋友圈（2）.m4a
 - 39_花絮1.m4a
 - 40_花絮2.m4a
 - 41_花絮3.m4a


# Option B：合并成一个 .m4b + 章节（chapter markers）

In [50]:
OUT_B = Path(ALBUM)
OUT_B.mkdir(exist_ok=True)

concat_list = OUT_B / "concat_list.txt"
with concat_list.open("w", encoding="utf-8") as f:
    for p in files:
        abs_path = p.resolve().as_posix()
        abs_path = abs_path.replace("'", r"'\''")
        f.write(f"file '{abs_path}'\n")

merged_tmp = OUT_B / "merged_tmp.m4b"

cmd = [
    FFMPEG, "-y",
    "-f", "concat", "-safe", "0",
    "-i", str(concat_list),
    "-c", "copy",
    str(merged_tmp)
]
print("Merging...")
subprocess.run(cmd, check=True)
print(f"✅ merged tmp: {merged_tmp.resolve()}")

Merging...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x7cf418280] Detected creation time before 1970, parsing as unix timestamp.
Input #0, concat, from '再世权臣_第2季_天谢/concat_list.txt':
  Duration: N/A, start: -0.036281, bitrate: 64

✅ merged tmp: /Users/jijiko/Desktop/GitHub/tools/m4a_to_audiobook/再世权臣_第2季_天谢/merged_tmp.m4b


    Last message repeated 2 times
[out#0/ipod @ 0x7ce80c780] video:0KiB audio:599115KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 2.154580%
size=  612023KiB time=21:18:03.27 bitrate=  65.4kbits/s speed=1.06e+04x elapsed=0:00:07.21    


In [51]:
def duration_seconds(path: Path) -> float:
    cmd = [
        FFPROBE, "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "format=duration",
        "-of", "json",
        str(path)
    ]
    out = subprocess.check_output(cmd)
    data = json.loads(out)
    return float(data["format"]["duration"])

durations = [duration_seconds(p) for p in files]
print("First 5 durations (sec):", durations[:5])
print(f"Total: {sum(durations)/3600:.2f} hours")

First 5 durations (sec): [2327.08, 3189.631995, 2792.71, 2543.165011, 2722.840998]
Total: 21.30 hours


In [52]:
ffmeta = OUT_B / "chapters.ffmeta"

start_ms = 0
lines = []
lines.append(";FFMETADATA1")

# 这些是章节文件里的元数据（会被写入容器）
lines.append(f"title={BOOK_TITLE}")
lines.append(f"artist={AUTHOR}")
lines.append(f"album={ALBUM}")

for p, d in zip(files, durations):
    end_ms = start_ms + int(round(d * 1000))
    chapter_title = p.stem

    lines.append("[CHAPTER]")
    lines.append("TIMEBASE=1/1000")
    lines.append(f"START={start_ms}")
    lines.append(f"END={end_ms}")
    lines.append(f"title={chapter_title}")

    start_ms = end_ms

ffmeta.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✅ ffmeta: {ffmeta.resolve()}")

final_m4b = OUT_B / f"{BOOK_TITLE}.m4b"

cmd = [
    FFMPEG, "-y",
    "-i", str(merged_tmp),
    "-i", str(COVER),
    "-f", "ffmetadata", "-i", str(ffmeta),

    "-map", "0:a:0",
    "-map", "1:v:0",

    # 章节 & 元数据来自 ffmeta（第2个输入索引=2）
    "-map_metadata", "2",
    "-map_chapters", "2",

    "-c:a", "copy",
    "-c:v", "mjpeg",
    "-disposition:v:0", "attached_pic",

    str(final_m4b)
]

print("Writing final m4b with chapters + cover + metadata...")
subprocess.run(cmd, check=True)
print(f"✅ Option B done: {final_m4b.resolve()}")

✅ ffmeta: /Users/jijiko/Desktop/GitHub/tools/m4a_to_audiobook/再世权臣_第2季_天谢/chapters.ffmeta
Writing final m4b with chapters + cover + metadata...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '再世权臣_第2季_天谢/merged_tmp.m4b':
  Metadata:
    major_brand     : M4A 
    minor_version   : 512
    compatible_brands: M4A isomiso2
    encoder         : Lavf62.3.1

✅ Option B done: /Users/jijiko/Desktop/GitHub/tools/m4a_to_audiobook/再世权臣_第2季_天谢/再世权臣_第2季_天谢.m4b


[out#0/ipod @ 0x8aec14300] video:204KiB audio:599115KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 2.154626%
frame=    1 fps=0.2 q=9.4 Lsize=  612232KiB time=00:00:00.04 bitrate=125385090.2kbits/s speed=0.00874x elapsed=0:00:04.57    
